# Демо 2. Шаблони розмови і підготовка даних для SFT

В демо 1 донавчання моделі було повʼязане з заміною голови, а отже частковою зміною архітектури нейронів. В багатьох випадках при донавчанні LLM моделей архітектура не змінюється, а змінюється радше формат введених і виведених данних. Для цього потрібно зрозуміти, **у якому вигляді модель бачить діалог**.
В будь-якому випадку фізично на вхід моделі подається тензор з токенами, а за те як з цих токенів утворюються семантичні повідомлення відповідають **шаблони розмови** (chat templates).
Під час донавчання модель може вивчити новий шаблон розмови.

Донавчання LLM методом SFT виконується на діалогах, відформатованих шаблоном розмови.
У цьому ноутбуці розглянемо, що таке шаблон розмови і як підготувати дані у форматі діалогів;
саме донавчання виконаємо в демо 3.

План:
1. Завантажити base- та instruct-версії SmolLM3.
2. Подивитися, як шаблон розмови перетворює різні типи діалогів на текст.
3. Порівняти відповіді base- та instruct-моделі.
4. Перевірити режими міркування `/think` і `/no_think`.
5. Порівняти шаблони різних моделей.
6. Підготувати датасет для SFT у форматі `messages`.

Демо за мотивами вправ курсу [smol-course](https://huggingface.co/learn/smol-course) і розділу 11
[Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter11/1).
Для запуску в Colab оберіть середовище з GPU: *Runtime → Change runtime type → T4 GPU*.

In [1]:
# Встановлюємо потрібні бібліотеки; -q вимикає зайвий вивід під час встановлення
!pip install -q "transformers>=4.55.0" "datasets" "accelerate"

In [2]:
import warnings

import torch
from datasets import get_dataset_split_names, load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

if torch.cuda.is_available():
    device = "cuda"
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Памʼять GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif torch.backends.mps.is_available():
    device = "mps"
    print("Apple MPS")
else:
    device = "cpu"
    warnings.warn("Обчислення на CPU: генерація буде дуже повільною, потрібна GPU")

GPU: Tesla T4
Памʼять GPU: 15.6 GB


In [3]:
# Детальна інформація про GPU NVIDIA: модель, зайнята й вільна памʼять
if device == "cuda":
    !nvidia-smi

Wed Nov 18 10:25:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Завантаження моделей

[**SmolLM3**](https://huggingface.co/blog/smollm3) — відкрита модель від Hugging Face на 3 млрд параметрів. Вона має дві версії:
- **Base** — лише після попереднього тренування: продовжує текст, але не веде діалог;
- **Instruct** (`SmolLM3-3B`) — донавчена виконувати інструкції й вести діалог.

In [4]:
base_model_name = "HuggingFaceTB/SmolLM3-3B-Base"
instruct_model_name = "HuggingFaceTB/SmolLM3-3B"

base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
instruct_tokenizer = AutoTokenizer.from_pretrained(instruct_model_name)

# Ваги SmolLM3 збережено в bfloat16, але T4 не підтримує його апаратно,
# тому завантажуємо у float16.
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name, dtype=torch.float16, device_map="auto"
)
instruct_model = AutoModelForCausalLM.from_pretrained(
    instruct_model_name, dtype=torch.float16, device_map="auto"
)
print("Моделі завантажено")

config.json:   0%|          | 0.00/943 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/151 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

... (12 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/182 [00:00<?, ?B/s]

Моделі завантажено


## 2. Як шаблон розмови форматує діалог

Діалог задається списком повідомлень із ролями:
- `system` — інструкції для моделі (роль, стиль, обмеження);
- `user` — повідомлення користувача;
- `assistant` — відповіді моделі.

Метод `apply_chat_template` перетворює цей список на рядок зі спеціальними токенами.

Параметр `add_generation_prompt`:
- `False` — для передачі в модель завершених діалогів, тобто для **тренувальних даних**;
- `True` — додає в кінець заголовок відповіді асистента, щоб модель почала відповідати; використовується для **інференсу**.

Для SmolLM3 у системне повідомлення можна додати прапорець `/think` або `/no_think`,
який вмикає чи вимикає режим міркування.

In [5]:
# Різні типи діалогів для порівняння
conversations = {
    "simple_qa": [
        {"role": "system", "content": "/no_think"},
        {"role": "user", "content": "What is machine learning?"},
    ],
    "with_system": [
        {
            "role": "system",
            "content": "You are a helpful AI assistant specialized in explaining technical concepts clearly. /no_think",
        },
        {"role": "user", "content": "What is machine learning?"},
    ],
    "multi_turn": [
        {"role": "system", "content": "You are a math tutor. /no_think"},
        {"role": "user", "content": "What is calculus?"},
        {
            "role": "assistant",
            "content": "Calculus is a branch of mathematics that deals with rates of change and accumulation of quantities.",
        },
        {"role": "user", "content": "Can you give me a simple example?"},
    ],
    "reasoning_task": [
        {"role": "system", "content": "/think"},
        {
            "role": "user",
            "content": "Solve step by step: If a train travels 120 miles in 2 hours, what is its average speed?",
        },
    ],
}

for conv_type, messages in conversations.items():
    print(f"--- {conv_type.upper()} ---")

    # Завершений діалог: так виглядають тренувальні дані
    formatted_complete = instruct_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    # Діалог для інференсу: у кінці додано заголовок відповіді асистента
    formatted_prompt = instruct_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    print("Завершений діалог (add_generation_prompt=False):")
    print(formatted_complete)
    print("\nДля інференсу (add_generation_prompt=True):")
    print(formatted_prompt)
    print("\n" + "=" * 50 + "\n")

--- SIMPLE_QA ---
Завершений діалог (add_generation_prompt=False):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 18 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face.

<|im_start|>user
What is machine learning?<|im_end|>


Для інференсу (add_generation_prompt=True):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 18 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face.

<|im_start|>user
What is machine learning?<|im_end|>
<|im_start|>assistant
<think>

</think>



--- WITH_SYSTEM ---
Завершений діалог (add_generation_prompt=False):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 18 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant specialized in explaining technical concepts clearly.

<|im_s

**Що варто помітити:**
- Шаблон SmolLM3 сам додає до системного повідомлення блок метаданих (дата відсічення знань, поточна дата, режим міркування)
  і стандартну інструкцію, якщо власної немає. Модель бачить більше, ніж ми написали.
- Кожне повідомлення обгорнуте токенами `<|im_start|>роль` … `<|im_end|>` (формат ChatML).
- З `add_generation_prompt=True` рядок закінчується на `<|im_start|>assistant`: з цього місця модель починає генерувати.

## 3. Base проти Instruct

Ставимо обом моделям одне питання. Base-модель отримує сирий текст без шаблону,
instruct-модель — діалог, відформатований шаблоном розмови.

In [6]:
def generate(model, tokenizer, text, max_new_tokens=150, temperature=0.7):
    """Генерує продовження тексту й повертає лише нові токени, без промпту."""
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Відкидаємо токени промпту: декодуємо лише те, що згенерувала модель
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [7]:
test_prompt = "Explain me, what is quantum computing?"

# Base-модель: сирий текст без шаблону
print("🤖 BASE-МОДЕЛЬ:")
print("-" * 50)
print(generate(base_model, base_tokenizer, test_prompt))

# Instruct-модель: діалог, відформатований шаблоном розмови
instruct_messages = [
    {"role": "system", "content": "/no_think"},
    {"role": "user", "content": test_prompt},
]
instruct_formatted = instruct_tokenizer.apply_chat_template(
    instruct_messages, tokenize=False, add_generation_prompt=True
)
print("\n" + "=" * 50)
print("💬 INSTRUCT-МОДЕЛЬ:")
print("-" * 50)
print(generate(instruct_model, instruct_tokenizer, instruct_formatted))

🤖 BASE-МОДЕЛЬ:
--------------------------------------------------


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


 How can it be different from conventional computing?
Quantum computing is a kind of a computing technology and it is based on the quantum physics. It uses the quantum states of matter to perform computational tasks. It uses the quantum phenomena to perform the tasks as it uses the quantum mechanics and algorithms.
It uses the quantum phenomena to perform the tasks such as superposition, state entanglement and quantum interference. It uses the quantum bits as the basic unit of data.
It uses the qubits as the basic unit of data. It uses the qubits for the computation. It uses the qubits for the computation and it can be one or zero at the same time. It can be one or zero at the same time. Conventional computing uses the bits

💬 INSTRUCT-МОДЕЛЬ:
--------------------------------------------------
Quantum computing is a fascinating field that leverages the principles of quantum mechanics to process information in ways that are fundamentally different from classical computers. Here's a simp

## 4. Режими міркування `/think` і `/no_think`

SmolLM3 — гібридна модель: у режимі `/think` вона спочатку генерує міркування в блоці `<think>…</think>`,
а потім відповідь; у режимі `/no_think` блок міркувань порожній.

In [8]:
reasoning_prompts = [
    "What is 15 × 24? Show your work.",
    "A recipe calls for 2 cups of flour for 12 cookies. How much flour is needed for 30 cookies?",
    "If I have $50 and spend $18.75 on lunch and $12.30 on a book, how much money do I have left?",
]

for thinking_flag in ["/no_think", "/think"]:
    print(f"===== Режим: {thinking_flag} =====\n")
    for i, prompt in enumerate(reasoning_prompts, 1):
        messages = [
            {"role": "system", "content": thinking_flag},
            {"role": "user", "content": prompt},
        ]
        formatted = instruct_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        # Нижча температура робить відповіді на арифметичні задачі стабільнішими
        answer = generate(instruct_model, instruct_tokenizer, formatted, max_new_tokens=300, temperature=0.3)
        print(f"Задача {i}: {prompt}")
        print(f"Відповідь: {answer}")
        print("\n" + "-" * 50 + "\n")

===== Режим: /no_think =====

Задача 1: What is 15 × 24? Show your work.
Відповідь: To solve the multiplication problem 15 × 24, we can use the standard multiplication algorithm. Here's how to do it step by step:

First, we multiply 15 by the units digit of 24, which is 4:
15 × 4 = 60

Next, we multiply 15 by the tens digit of 24, which is 2. Since this digit is in the tens place, we need to shift the result one position to the left (i.e., multiply by 10):
15 × 2 × 10 = 30 × 10 = 300

Now, we add the two results together:
60 + 300 = 360

So, the answer to 15 × 24 is 360.

--------------------------------------------------

Задача 2: A recipe calls for 2 cups of flour for 12 cookies. How much flour is needed for 30 cookies?
Відповідь: To solve this problem, we need to determine the amount of flour needed per cookie and then scale it up to 30 cookies.

Step 1: Calculate the amount of flour per cookie.
The recipe calls for 2 cups of flour for 12 cookies. To find the amount of flour per co

## 5. У кожної моделі свій шаблон

Ролі `system`, `user`, `assistant` — це домовленість на рівні формату `messages`.
У токенах кожна модель записує їх по-своєму. Порівняємо шаблони кількох моделей.

In [9]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hi!"},
    {"role": "assistant", "content": "Hello! How can I help?"},
]

for name in ["HuggingFaceTB/SmolLM3-3B", "Qwen/Qwen3-0.6B", "microsoft/Phi-3.5-mini-instruct"]:
    tok = AutoTokenizer.from_pretrained(name)
    print(f"===== {name} =====")
    print(tok.apply_chat_template(messages, tokenize=False))
    print()

===== HuggingFaceTB/SmolLM3-3B =====
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 18 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a helpful assistant.

<|im_start|>user
Hi!<|im_end|>
<|im_start|>assistant
Hello! How can I help?<|im_end|>




config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

... (7 outputs left out)

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

===== microsoft/Phi-3.5-mini-instruct =====
<|system|>
You are a helpful assistant.<|end|>
<|user|>
Hi!<|end|>
<|assistant|>
Hello! How can I help?<|end|>
<|endoftext|>



Прапорці режиму міркування також залежать від моделі. У Qwen3, наприклад,
режим вмикається параметром `enable_thinking` у `apply_chat_template`.

In [10]:
qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
question = [{"role": "user", "content": "What is 2 + 2?"}]

for enable_thinking in [True, False]:
    print(f"===== enable_thinking={enable_thinking} =====")
    print(qwen_tokenizer.apply_chat_template(
        question, tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking
    ))

===== enable_thinking=True =====
<|im_start|>user
What is 2 + 2?<|im_end|>
<|im_start|>assistant

===== enable_thinking=False =====
<|im_start|>user
What is 2 + 2?<|im_end|>
<|im_start|>assistant
<think>

</think>




### А що з base-моделлю?

Base-модель не донавчали на діалогах, тому шаблон у її токенізаторі або відсутній, або модель ніколи не бачила його токенів у ролі розмітки діалогу.
Перед SFT шаблон для base-моделі задають самостійно — найпростіше взяти його з instruct-версії тієї самої моделі.

In [11]:
print("Шаблон є в base-токенізаторі:  ", base_tokenizer.chat_template is not None)
print("Шаблон є в instruct-токенізаторі:", instruct_tokenizer.chat_template is not None)

Шаблон є в base-токенізаторі:   False
Шаблон є в instruct-токенізаторі: True


## 6. Підготовка датасету для SFT

Для SFT дані зберігають у форматі `messages` — тобто у списку повідомлень із ролями.
Бібліотека TRL (`SFTTrainer`) сама застосовує до нього шаблон розмови під час тренування.

### SmolTalk2

**SmolTalk2** — датасет, на якому донавчали SmolLM3. Він складається з багатьох частин (splits) з різних джерел.
Список частин можна отримати без завантаження самих даних.

In [12]:
splits = get_dataset_split_names("HuggingFaceTB/smoltalk2", "SFT")
print(f"Кількість частин: {len(splits)}")
for s in splits:
    print(" -", s)

README.md:   0%|          | 0.00/23.5k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/124 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Кількість частин: 25
 - LongAlign_64k_Qwen3_32B_yarn_131k_think
 - OpenThoughts3_1.2M_think
 - aya_dataset_Qwen3_32B_think
 - multi_turn_reasoning_if_think
 - s1k_1.1_think
 - smolagents_toolcalling_traces_think
 - smoltalk_multilingual8_Qwen3_32B_think
 - smoltalk_systemchats_Qwen3_32B_think
 - table_gpt_Qwen3_32B_think
 - LongAlign_64k_context_lang_annotated_lang_6_no_think
 - Mixture_of_Thoughts_science_no_think
 - OpenHermes_2.5_no_think
 - OpenThoughts3_1.2M_no_think_no_think
 - hermes_function_calling_v1_no_think
 - smoltalk_multilingual_8languages_lang_5_no_think
 - smoltalk_smollm3_everyday_conversations_no_think
 - smoltalk_smollm3_explore_instruct_rewriting_no_think
 - smoltalk_smollm3_smol_magpie_ultra_no_think
 - smoltalk_smollm3_smol_rewrite_no_think
 - smoltalk_smollm3_smol_summarize_no_think
 - smoltalk_smollm3_systemchats_30k_no_think
 - table_gpt_no_think
 - tulu_3_sft_personas_instruction_following_no_think
 - xlam_traces_no_think
 - smoltalk_everyday_convs_reasoning_

In [13]:
# Завантажуємо лише файл однієї частини через data_files: повний датасет займає десятки гігабайт,
# а load_dataset(..., split=...) спершу перебирає файли всіх частин і працює дуже довго
split = "smoltalk_everyday_convs_reasoning_Qwen3_32B_think"
everyday = load_dataset("HuggingFaceTB/smoltalk2", data_files=f"SFT/{split}-*.parquet", split="train")
print(everyday)
print()
for message in everyday[0]["messages"]:
    print(f"[{message['role']}] {message['content'][:200]}")

SFT/smoltalk_everyday_convs_reasoning_Qw(…): reconstructing file:   0%|          |  0.00B / 6.33MB            

SFT/smoltalk_everyday_convs_reasoning_Qw(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['messages', 'chat_template_kwargs', 'source'],
    num_rows: 2057
})

[user] Hi there
[assistant] <think>
Okay, the user sent "Hi there". That's a friendly greeting. I should respond in a welcoming way. Let me check the guidelines. I need to be helpful, keep the conversation going, and maybe ask h
[user] I'm looking for a healthy breakfast idea. What's a good option?
[assistant] <think>
Okay, the user is asking for a healthy breakfast idea. Let me think about what makes a breakfast healthy. It should be balanced, providing a mix of nutrients like protein, fiber, healthy fats,


Ця частина зібрана для режиму `/think`, тому відповіді асистента містять блок міркувань `<think>…</think>`.

Як видно з назви частини, відповіді в ній згенерувала модель Qwen3-32B. Такі **синтетичні дані** від сильнішої моделі —
поширений спосіб отримати тисячі прикладів для SFT без ручного написання відповідей. Помилки моделі-генератора при цьому переходять у дані,
тому синтетичні дані фільтрують.

### Перетворення власного датасету у формат `messages`

Більшість датасетів має інший формат, наприклад колонки «питання» і «відповідь».
Перетворимо датасет задач з математики **GSM8K** у формат `messages`.

In [14]:
gsm8k = load_dataset("openai/gsm8k", "main", split="train[:100]")  # невелика вибірка для демо
print("Оригінальний приклад:", gsm8k[0])


def to_messages(examples):
    """Перетворює пари «питання — відповідь» на діалоги з ролями."""
    processed = []
    for question, answer in zip(examples["question"], examples["answer"]):
        processed.append([
            {"role": "system", "content": "You are a math tutor. Solve problems step by step."},
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ])
    return {"messages": processed}


gsm8k_messages = gsm8k.map(to_messages, batched=True, remove_columns=gsm8k.column_names)
print("\nУ форматі messages:", gsm8k_messages[0])

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Оригінальний приклад: {'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}


Map:   0%|          | 0/100 [00:00<?, ? examples/s]


У форматі messages: {'messages': [{'content': 'You are a math tutor. Solve problems step by step.', 'role': 'system'}, {'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'role': 'user'}, {'content': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72', 'role': 'assistant'}]}


### Що побачить модель під час тренування

Застосуємо шаблон розмови до підготовленого прикладу. Саме цей рядок модель отримає під час SFT.

In [15]:
text = instruct_tokenizer.apply_chat_template(
    gsm8k_messages[0]["messages"],
    tokenize=False,
    add_generation_prompt=False,  # діалог завершений: відповідь асистента вже є
)
print(text)
print(f"Довжина в токенах: {len(instruct_tokenizer(text)['input_ids'])}")

<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 18 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a math tutor. Solve problems step by step.

<|im_start|>user
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>
<|im_start|>assistant
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72<|im_end|>

Довжина в токенах: 145


**Що варто помітити:** шаблон знову додав метадані й режим `/think`, хоча в даних їх немає,
а відповідь асистента не містить блоку міркувань. Якщо донавчати на таких прикладах, модель вчитиметься
відповідати без міркувань навіть у режимі `/think`. Тому формат даних і шаблон мають узгоджуватися з тим,
як модель використовуватиметься після донавчання.

---
# Відповіді на теоретичні питання 0.1 (питання 11–21)

### 11. Як довго виконувалися клітинки ноутбука 02_chat_templates? Чи були проблеми?

Ноутбук запускався в Google Colab на GPU **Tesla T4 (15.6 GB)**.

| Клітинка | Час |
|---|---|
| Встановлення бібліотек | 10 с |
| Імпорти, перевірка GPU | 21 с |
| `nvidia-smi` | < 1 с |
| **Завантаження двох моделей SmolLM3-3B (base та instruct, по ~6 GB)** | **≈ 2 хв** |
| Секція 2: шаблони розмов | < 1 с |
| Секція 3: base проти instruct | 15 с |
| **Секція 4: `/think` і `/no_think` (6 генерацій)** | **57 с** |
| Секція 5: шаблони трьох моделей | 11 с |
| Секція 6: SmolTalk2, GSM8K | ≈ 10 с |

Усього ≈ 4–5 хвилин, найдовше — завантаження моделей.

**Проблеми:**
- спершу Colab не давав запустити ноутбук через помилку **«Слишком много сеансов»**: безкоштовна версія дозволяє лише один сеанс з GPU, а попередній ноутбук ще працював. Допомогло завершення старого сеансу через *Среда выполнения → Управление сеансами*;
- під час виконання помилок не було, лише попередження: `unauthenticated requests to the HF Hub` (завантаження без токена Hugging Face), `clean_up_tokenization_spaces` (налаштування токенізатора) і `rope_parameters` для Phi-3.5. На результат вони не вплинули.

### 12. Для чого змінювати add_generation_prompt? Як за згенерованими розмовами дізнатися його значення?

`add_generation_prompt` визначає, чи додавати **в кінець** рядка заголовок відповіді асистента:
- **`True` — для інференсу (використання моделі).** У кінець додається `<|im_start|>assistant`. Так модель «бачить», що зараз її черга відповідати, і починає генерувати саме відповідь асистента. Без цього заголовка модель могла б продовжити репліку користувача або почати новий хід за користувача.
- **`False` — для тренувальних даних.** Діалог уже завершений, відповідь асистента є в даних, тому зайвий порожній заголовок у кінці не потрібен. Інакше модель навчалася б на «обірваних» прикладах.

**Як визначити значення за виводом:**
- рядок **закінчується на `<|im_start|>assistant`** (для `/no_think` ще й з порожнім блоком `<think>` `</think>`), і після нього немає тексту → `add_generation_prompt=True`;
- рядок **закінчується на `<|im_end|>`** (останнім іде повідомлення користувача чи асистента) → `add_generation_prompt=False`.

Наприклад, для `REASONING_TASK` завершений діалог закінчується на `…average speed?<|im_end|>`, а версія для інференсу — на `<|im_start|>assistant`.

### 13. Чим відрізняються ролі system і assistant?

| | `system` | `assistant` |
|---|---|---|
| **Хто «говорить»** | розробник застосунку | сама модель |
| **Що містить** | інструкції: роль моделі, стиль, обмеження, режим (`/think`, `/no_think`) | відповіді моделі користувачу |
| **Де в діалозі** | один раз, на самому початку | після кожного повідомлення `user` |
| **Чи генерує модель** | ні, модель лише читає його й виконує | так, це те, що модель пише |
| **Під час SFT** | зазвичай не входить у loss | саме на цих токенах модель вчиться (`assistant_only_loss`) |

`system` задає **правила для всієї розмови** («You are a math tutor»), і користувач його зазвичай не бачить. `assistant` — це **репліки моделі**. У тренувальних даних вони є прикладами правильних відповідей, а в багатокроковому діалозі (як `multi_turn`) — історією попередніх відповідей, яку модель враховує.

### 14. Коментар до виводу секції 2: які частини додав шаблонізатор і навіщо?

Ми передали лише кілька коротких повідомлень, а шаблон SmolLM3 сформував значно довший рядок. Він додав:

1. **Спеціальні токени ролей (формат ChatML):** кожне повідомлення обгорнуто в `<|im_start|>роль` … `<|im_end|>`. Так модель розрізняє, де закінчується одна репліка й починається інша і хто її автор.
2. **Блок `## Metadata`** у системному повідомленні:
   - `Knowledge Cutoff Date: June 2025` — до якої дати модель знає світ;
   - `Today Date: 18 November 2026` — поточна дата (шаблон підставляє її автоматично);
   - `Reasoning Mode: /think` або `/no_think` — режим міркування.
   Це дає моделі контекст дати й явно повідомляє режим роботи.
3. **Блок `## Custom Instructions`** з нашим системним текстом («You are a math tutor.»). Прапорець `/no_think` з нього прибрано й перенесено в метадані. Якщо власного тексту немає (як у `simple_qa` чи `reasoning_task`), шаблон підставляє **стандартну інструкцію**: «You are a helpful AI assistant named SmolLM, trained by Hugging Face…». Тобто модель отримує роль, навіть коли ми її не задали.
4. **Порожній блок `<think>` `</think>`** перед відповіддю асистента в режимі `/no_think`, зокрема в попередніх відповідях асистента в `multi_turn`. Так модель «бачить», що етап міркування вже пройдено, і одразу пише відповідь.
5. **Заголовок `<|im_start|>assistant` у кінці**, лише при `add_generation_prompt=True` (див. питання 12).

**Висновок:** модель бачить значно більше, ніж ми написали. Це важливо знати і для інференсу, і особливо для тренування: дані мають бути в тому самому форматі.

### 15. Різниця між Base і Instruct (секція 3)

На те саме питання «Explain me, what is quantum computing?»:

**Base-модель** (сирий текст без шаблону):
- спершу **продовжила питання ще одним питанням**: «How can it be different from conventional computing?», ніби це текст зі списку питань чи форуму;
- далі написала пояснення, але дуже **повторюване й без структури**: «It uses the quantum states… It uses the quantum phenomena… It uses the qubits… It uses the qubits…».
- Вона не «відповідає» користувачу, а просто **продовжує текст** так, як він найімовірніше продовжувався б в інтернеті.

**Instruct-модель** (діалог через шаблон розмови):
- одразу **відповідає як асистент**: «Quantum computing is a fascinating field that leverages the principles of quantum mechanics… Here's a simplified explanation»;
- відповідь **структурована**: заголовок «Quantum Computing Basics», нумерований список (1. Qubits, 2. Entanglement…);
- текст зв'язний, без повторів, звернений до користувача. Відповідь обірвана лише через ліміт `max_new_tokens=150`.

**Висновок:** знання в обох моделях однакові, адже instruct-модель створена з base донавчанням. Але лише instruct-модель після SFT на діалогах уміє **виконувати інструкцію й вести розмову** у форматі шаблону. Base-модель для чат-застосунку не підходить без донавчання.

### 16. Чому в режимі /think передано max_new_tokens=300?

У режимі `/think` модель спершу пише **міркування** в блоці `<think>…</think>`, і лише потім відповідь. Міркування займають значно більше токенів, ніж сама відповідь, тож ліміт за замовчуванням (150 у функції `generate`) збільшено до **300**, щоб модель мала більше місця. Водночас ліміт не роблять дуже великим, щоб демо працювало швидко: кожен токен генерується окремим кроком, і 6 генерацій і так зайняли ≈ 1 хвилину.

**Але в моєму запуску 300 токенів виявилося замало:** в усіх трьох задачах у режимі `/think` модель **не дійшла до фінальної відповіді**. Генерація обірвалася посеред міркувань («…15 × 4. That», «…5 cups of flour for 30 cookies. But», «Starting from the rightmost digit: 0 - 5. Hmm, can't»). Модель «думає» багатослівно: перевіряє себе кількома способами («Wait, let me check that again»). Для реального використання режиму `/think` потрібен ліміт ~1000–2000 токенів.

### 17. Чи впоралася модель з математичними задачами? Який режим обрати?

| Задача | Правильна відповідь | `/no_think` | `/think` (300 токенів) |
|---|---|---|---|
| 15 × 24 | 360 | ✅ **360**: 15 × 4 = 60, 15 × 20 = 300, 60 + 300 = 360 | у міркуваннях порахувала 360 і почала перевіряти іншим способом, але **відповідь не дописала** (обрізано) |
| Борошно на 30 печив | 5 чашок | ✅ **5 чашок**: 2/12 = 1/6 чашки на печиво, 1/6 × 30 = 5 | у міркуваннях отримала 5 чашок, але **відповідь не дописала** |
| $50 − $18.75 − $12.30 | $18.95 | ✅ **$18.95**: витрачено $31.05, лишилось $18.95 | порахувала витрати $31.05, але обірвалася на відніманні («0 - 5. Hmm, can't») |

**Висновок:** у режимі `/no_think` модель **правильно розв'язала всі три задачі**, причому все одно показала хід розв'язку покроково, просто без окремого блоку міркувань. У режимі `/think` міркування теж правильні, але модель багатослівна: перевіряє себе кількома способами («Wait, let me check that again»). Через ліміт 300 токенів **жодна відповідь не була доведена до кінця**, тож користувач не отримав би відповіді взагалі.

**Який режим обрала б я:**
- для **простих задач** (арифметика, задачі на 2–3 дії, як ці) — **`/no_think`**: відповідь швидка (у кілька разів менше токенів), правильна, і хід розв'язку видно;
- для **складних задач** (багатокрокові задачі, доведення, задачі з підступом) — **`/think`**, але з великим лімітом токенів (1000+), щоб модель встигла і подумати, і дописати відповідь. Міркування з самоперевіркою зменшують кількість помилок, хоч і коштують часу.

### 18. Які відмінності в шаблонах розмов у секції 5?

Той самий діалог (system / user / assistant) три моделі записують по-різному:

| | SmolLM3-3B | Qwen3-0.6B | Phi-3.5-mini-instruct |
|---|---|---|---|
| Токени ролей | `<|im_start|>role` … `<|im_end|>` (ChatML) | `<|im_start|>role` … `<|im_end|>` (ChatML) | `<|system|>`, `<|user|>`, `<|assistant|>`, кінець репліки `<|end|>` |
| Що додається до system | блок `## Metadata` (дата відсічення знань, сьогоднішня дата, `Reasoning Mode: /think`) і `## Custom Instructions` | нічого, системний текст як є | нічого |
| Міркування | режим `/think` у метаданих (за замовчуванням увімкнений) | порожній блок `<think>` `</think>` перед відповіддю асистента | немає режиму міркування |
| Кінець діалогу | `<|im_end|>` | `<|im_end|>` | `<|end|>` і додатково `<|endoftext|>` |

Також відрізняється **керування режимом міркування**: у SmolLM3 — прапорцями `/think` / `/no_think` у системному повідомленні, у Qwen3 — параметром `enable_thinking` у `apply_chat_template`. При `enable_thinking=False` шаблон Qwen3 одразу додає порожній блок `<think>` `</think>` після заголовка асистента, і модель пропускає міркування. При `True` рядок закінчується просто на `<|im_start|>assistant`.

У **base-токенізатора** SmolLM3 шаблону немає взагалі (`chat_template is not None` → `False`), а в instruct є.

**Висновок:** ролі `system/user/assistant` — спільна домовленість лише на рівні формату `messages`. У токенах кожна модель має свій формат, і модель розуміє лише той, на якому її тренували.

### 19. Для чого знати про шаблони розмов, якщо ми хочемо донавчити модель за допомогою SFT?

1. **Формат під час тренування і використання має збігатися.** Під час SFT модель вчиться на рядках, які формує шаблон. Якщо потім подавати їй запити в іншому форматі (інші токени ролей, без метаданих), вона працюватиме гірше або взагалі не розпізнає, де її черга відповідати.
2. **Кожна модель має свій шаблон** (секція 5). Для instruct-моделі треба використовувати її рідний шаблон, бо його токени вона вже знає. Для **base-моделі** шаблону немає, і перед SFT його треба задати самостійно, найкраще взяти з instruct-версії тієї ж моделі.
3. **Шаблон додає приховані частини**: метадані, стандартні інструкції, режим `/think`, порожні блоки `<think>`. Якщо дані цього не враховують, модель вчиться не тому, що ми хотіли. У секції 6 відповіді GSM8K без міркувань потрапили в діалог із режимом `/think`, і модель вчилася б відповідати без міркувань навіть у режимі `/think`.
4. **Loss лише на відповідях асистента** (`assistant_only_loss`): бібліотека за токенами шаблону знаходить, де відповіді асистента, і рахує помилку лише на них. Без правильного шаблону це не спрацює.
5. **Правильний `add_generation_prompt`:** `False` для тренувальних даних, `True` для інференсу.

### 20. З яких даних побудовано датасет HuggingFaceTB/smoltalk2?

Частина SFT датасету SmolTalk2 складається з **25 частин (splits)**, кожна з окремого джерела. Назви поділені на `_think` (з міркуваннями) і `_no_think` (без міркувань), щоб навчити модель обох режимів. Джерела:

- **Синтетичні діалоги, згенеровані моделлю Qwen3-32B:** `smoltalk_everyday_convs_reasoning_Qwen3_32B_think` (повсякденні розмови з міркуваннями), `smoltalk_multilingual8_Qwen3_32B_think` (8 мов), `smoltalk_systemchats_Qwen3_32B_think`, `aya_dataset_Qwen3_32B_think`, `table_gpt_Qwen3_32B_think` (робота з таблицями);
- **Задачі з міркуваннями (математика, код, наука):** `OpenThoughts3_1.2M`, `s1k_1.1`, `Mixture_of_Thoughts_science`, `multi_turn_reasoning_if`;
- **Довгий контекст:** `LongAlign_64k_…` (тексти до 64–131 тис. токенів);
- **Виклик інструментів і агенти:** `smolagents_toolcalling_traces`, `hermes_function_calling_v1`, `xlam_traces`;
- **Загальні інструкції:** `OpenHermes_2.5`, `tulu_3_sft_personas_instruction_following`;
- **Власні дані SmolTalk для SmolLM3:** `smoltalk_smollm3_everyday_conversations`, `…_smol_rewrite` (переписування), `…_smol_summarize` (підсумовування), `…_systemchats_30k`, `…_smol_magpie_ultra`, `…_explore_instruct_rewriting`, `smoltalk_multilingual_8languages`.

Значна частина даних **синтетична**, тобто відповіді згенерувала сильніша модель (переважно Qwen3-32B). Так отримують тисячі прикладів без ручного написання. Завантажена частина `smoltalk_everyday_convs_reasoning_Qwen3_32B_think` містить 2 057 діалогів з колонками `messages`, `chat_template_kwargs` і `source`. Відповіді асистента в ній містять блок `<think>`.

### 21. Які модифікації даних відбуваються в секції «Перетворення власного датасету у формат messages»?

Беремо перші **100 прикладів** GSM8K (`train[:100]`) і функцією `to_messages` через `map(batched=True)` перетворюємо кожен приклад:

1. **Було:** два стовпці `question` і `answer`, тобто пара «питання — відповідь».
2. **Стало:** один стовпець `messages`, список із трьох повідомлень з ролями:
   - **додано нове** системне повідомлення `system`: «You are a math tutor. Solve problems step by step.», якого в оригінальних даних не було;
   - `question` → повідомлення `user`;
   - `answer` → повідомлення `assistant`.
3. **Оригінальні стовпці видалено** (`remove_columns=gsm8k.column_names`).

**Що не змінюється:** текст відповіді лишається як є, разом зі службовими позначками GSM8K: обчислення в `<<48/2=24>>` і фінальна відповідь після `#### 72`. Модель, донавчена на таких даних, теж писала б ці позначки.

**Що додає шаблон далі** (наступна клітинка): метадані й режим `Reasoning Mode: /think`, хоча відповідь не має блоку міркувань. Після шаблону приклад займає **145 токенів**. Як зазначено в ноутбуці, така невідповідність навчила б модель відповідати без міркувань навіть у режимі `/think`.